### Import library

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")
sns.set(style="whitegrid", palette="pastel", font_scale=1.1)

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, collect_set, struct
from pyspark.ml.fpm import FPGrowth


### Data API

In [ ]:
from ucimlrepo import fetch_ucirepo 
# fetch dataset 
online_retail = fetch_ucirepo(id=352) 
  
# data (as pandas dataframes) 
X = online_retail.data.original
X.head()




,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


### Cleaning data for dashboard

In [2]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import os
import pyarrow as pa
import pyarrow.parquet as pq

# Tạo thư mục đầu ra nếu chưa tồn tại
output_dir = 'online_retail_data'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Tải dữ liệu
print("Đang tải dữ liệu từ UCI ML Repository...")
online_retail = fetch_ucirepo(id=352)
X = online_retail.data.original

print(f"Kích thước dữ liệu ban đầu: {X.shape}")
print("Các cột trong dữ liệu:")
print(X.columns.tolist())

# Tiền xử lý dữ liệu
print("\nĐang tiền xử lý dữ liệu...")

# Chuyển đổi InvoiceDate sang kiểu datetime
X['InvoiceDate'] = pd.to_datetime(X['InvoiceDate'], errors='coerce')

# Loại bỏ các hàng có InvoiceDate null
X = X.dropna(subset=['InvoiceDate'])

# Chuyển đổi các cột số và loại bỏ giá trị không hợp lệ
X['Quantity'] = pd.to_numeric(X['Quantity'], errors='coerce')
X['UnitPrice'] = pd.to_numeric(X['UnitPrice'], errors='coerce')
X = X.dropna(subset=['Quantity', 'UnitPrice'])

# THAY ĐỔI 1: Chỉ loại bỏ hàng có Quantity < 0 (giữ lại Quantity = 0)
X = X[X['Quantity'] >= 0]
X = X[X['UnitPrice'] >= 0]

# THAY ĐỔI 2: Chuẩn hóa Description dựa trên StockCode
print("\nĐang chuẩn hóa Description dựa trên StockCode...")

# Tạo một bản sao của DataFrame ban đầu để giữ lại tất cả dữ liệu
X_original = X.copy()

# Lọc ra các bản ghi có Description không rỗng và có ý nghĩa để tạo bảng tra cứu
X_desc = X[~X['Description'].isna() & (X['Description'] != '')].copy()

print(f"Số lượng bản ghi có Description không rỗng: {X_desc.shape[0]}")

# Tạo DataFrame tần suất xuất hiện của mỗi cặp StockCode-Description
desc_counts = X_desc.groupby(['StockCode', 'Description']).size().reset_index(name='count')

# Sắp xếp theo StockCode và số lần xuất hiện (giảm dần)
desc_counts = desc_counts.sort_values(['StockCode', 'count'], ascending=[True, False])

# Lấy Description xuất hiện nhiều nhất cho mỗi StockCode
clean_descriptions = desc_counts.drop_duplicates(subset='StockCode', keep='first')[['StockCode', 'Description']]

print(f"Số lượng StockCode duy nhất có trong bảng tra cứu: {clean_descriptions.shape[0]}")

# Áp dụng Description chuẩn hóa cho toàn bộ DataFrame
# Loại bỏ cột Description hiện tại và join với cột Description đã chuẩn hóa
X_original = X_original.drop(columns=['Description'])
X_cleaned = pd.merge(X_original, clean_descriptions, on='StockCode', how='left')

# Kiểm tra xem có StockCode nào không có mô tả chuẩn hóa không
missing_desc = X_cleaned[X_cleaned['Description'].isna()]
print(f"Số lượng bản ghi không có mô tả sau khi join: {missing_desc.shape[0]}")

if missing_desc.shape[0] > 0:
    print("Một số StockCode không có Description chuẩn hóa. Đang gán giá trị mặc định...")
    # Điền giá trị mặc định cho các StockCode không có mô tả chuẩn hóa
    X_cleaned['Description'] = X_cleaned['Description'].fillna(f"Unknown Product")

X = X_cleaned

# Tạo cột TotalValue
X['TotalValue'] = X['Quantity'] * X['UnitPrice']

# Quan trọng: Chuyển đổi timestamp thành millisecond để tương thích với Spark
X['InvoiceDate'] = X['InvoiceDate'].dt.floor('ms')  # Làm tròn xuống millisecond

# Tạo các cột thời gian bổ sung
X['Year'] = X['InvoiceDate'].dt.year
X['Month'] = X['InvoiceDate'].dt.month
X['Day'] = X['InvoiceDate'].dt.day
X['Hour'] = X['InvoiceDate'].dt.hour
X['DayOfWeek'] = X['InvoiceDate'].dt.dayofweek

# Hiển thị dữ liệu mẫu sau xử lý
print("\nDữ liệu mẫu sau xử lý:")
print(X.head())

# Hiển thị thông tin về các StockCode có Description sau khi clean
print(f"\nSố lượng StockCode duy nhất trong dữ liệu sau khi xử lý: {X['StockCode'].nunique()}")
print(f"Số lượng bản ghi sau khi xử lý: {X.shape[0]}")

# Lưu thành file Parquet với cấu hình tương thích Spark
parquet_file = os.path.join(output_dir, 'online_retail_spark.parquet')

print(f"\nĐang lưu file Parquet tại: {os.path.abspath(parquet_file)}")
X.to_parquet(
    parquet_file,
    engine='pyarrow',
    compression='snappy',
    index=False,
    coerce_timestamps='ms',         # Sử dụng millisecond cho timestamp
    allow_truncated_timestamps=True  # Cho phép làm tròn timestamp
)

# Hiển thị kích thước file
file_size_mb = os.path.getsize(parquet_file) / (1024 * 1024)
print(f"File Parquet đã được tạo thành công! Kích thước: {file_size_mb:.2f} MB")
print(f"Đường dẫn: {os.path.abspath(parquet_file)}")

# Đọc lại và kiểm tra file Parquet
print("\nĐọc lại file để kiểm tra:")
df_check = pd.read_parquet(parquet_file)
print(df_check.head())
print(f"Số lượng bản ghi: {len(df_check)}")
print(f"Kiểu dữ liệu InvoiceDate: {df_check['InvoiceDate'].dtype}")

Đang tải dữ liệu từ UCI ML Repository...
Kích thước dữ liệu ban đầu: (541909, 8)
Các cột trong dữ liệu:
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

Đang tiền xử lý dữ liệu...

Đang chuẩn hóa Description dựa trên StockCode...
Số lượng bản ghi có Description không rỗng: 530691
Số lượng StockCode duy nhất có trong bảng tra cứu: 3925
Số lượng bản ghi không có mô tả sau khi join: 16
Một số StockCode không có Description chuẩn hóa. Đang gán giá trị mặc định...

Dữ liệu mẫu sau xử lý:
  InvoiceNo StockCode  Quantity         InvoiceDate  UnitPrice  CustomerID  \
0    536365    85123A         6 2010-12-01 08:26:00       2.55     17850.0   
1    536365     71053         6 2010-12-01 08:26:00       3.39     17850.0   
2    536365    84406B         8 2010-12-01 08:26:00       2.75     17850.0   
3    536365    84029G         6 2010-12-01 08:26:00       3.39     17850.0   
4    536365    84029E         6 2010-12-01 08:26:00       3.39 

In [4]:
# Kiểm tra giá trị của cột DayOfWeek
day_of_week_values = X['DayOfWeek'].unique()
print("\nCác giá trị độc nhất trong cột DayOfWeek:")
print(sorted(day_of_week_values))

# Kiểm tra cụ thể giá trị 5 (Thứ Bảy)
has_saturday = 5 in day_of_week_values
print(f"Dữ liệu có chứa Thứ Bảy (DayOfWeek = 5): {has_saturday}")

# Đếm số lượng bản ghi cho mỗi ngày trong tuần
day_counts = X['DayOfWeek'].value_counts().sort_index()
print("\nSố lượng bản ghi theo ngày trong tuần:")
for day, count in day_counts.items():
    day_name = ["Thứ Hai", "Thứ Ba", "Thứ Tư", "Thứ Năm", "Thứ Sáu", "Thứ Bảy", "Chủ Nhật"][int(day)]
    print(f"{day} ({day_name}): {count} bản ghi")


Các giá trị độc nhất trong cột DayOfWeek:
[0, 1, 2, 3, 4, 6]
Dữ liệu có chứa Thứ Bảy (DayOfWeek = 5): False

Số lượng bản ghi theo ngày trong tuần:
0 (Thứ Hai): 93308 bản ghi
1 (Thứ Ba): 99804 bản ghi
2 (Thứ Tư): 92559 bản ghi
3 (Thứ Năm): 101221 bản ghi
4 (Thứ Sáu): 80481 bản ghi
6 (Chủ Nhật): 63910 bản ghi


### Reading dashboard parquet data

In [3]:
"""
Đọc file Parquet với PySpark và thực hiện phân tích dữ liệu
"""

import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, count, countDistinct, avg, desc, date_format
from pyspark.sql.functions import year, month, dayofmonth, hour, dayofweek

# Thiết lập JAVA_HOME (điều chỉnh đường dẫn cho phù hợp với máy của bạn)
os.environ['JAVA_HOME'] = r'C:\Program Files\Java\jdk-20'  # Thay đổi nếu cần

# Khởi tạo Spark Session
print("Khởi tạo Spark Session...")
spark = SparkSession.builder \
    .appName("OnlineRetailAnalysis") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

# Đường dẫn đến file Parquet
parquet_file = "online_retail_data/online_retail_spark.parquet"

# Kiểm tra file tồn tại
import os.path
if not os.path.exists(parquet_file):
    print(f"Lỗi: File không tồn tại tại {os.path.abspath(parquet_file)}")
    exit(1)
else:
    print(f"Đọc file Parquet từ: {os.path.abspath(parquet_file)}")

# Đọc file Parquet
print("\nĐọc file Parquet...")
try:
    df = spark.read.parquet(parquet_file)
    print("Đọc file thành công!")
except Exception as e:
    print(f"Lỗi khi đọc file: {str(e)}")
    spark.stop()
    exit(1)

# 1. Hiển thị schema và dữ liệu mẫu
print("\n=== SCHEMA DỮ LIỆU ===")
df.printSchema()

print("\n=== 5 DÒNG DỮ LIỆU MẪU ===")
df.show(5)

# 2. Thống kê cơ bản
print("\n=== THỐNG KÊ CƠ BẢN ===")
print(f"Số lượng bản ghi: {df.count()}")
print(f"Số lượng cột: {len(df.columns)}")
type(df)

Khởi tạo Spark Session...
Đọc file Parquet từ: c:\Users\Admin\Desktop\big_data_project\online_retail_data\online_retail_spark.parquet

Đọc file Parquet...
Đọc file thành công!

=== SCHEMA DỮ LIỆU ===
root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- InvoiceDate: timestamp_ntz (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: double (nullable = true)
 |-- Country: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- TotalValue: double (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Day: integer (nullable = true)
 |-- Hour: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)


=== 5 DÒNG DỮ LIỆU MẪU ===
+---------+---------+--------+-------------------+---------+----------+--------------+--------------------+------------------+----+-----+---+----+---------+
|InvoiceNo|StockCode|Quantity|        Invo

pyspark.sql.dataframe.DataFrame

### Cleaning data for associate analysis

In [3]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import os
import pyarrow as pa
import pyarrow.parquet as pq

# Tạo thư mục đầu ra nếu chưa tồn tại
output_dir = 'associate_retail_data'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Tải dữ liệu
print("Đang tải dữ liệu từ UCI ML Repository...")
online_retail = fetch_ucirepo(id=352)
X = online_retail.data.original

print(f"Kích thước dữ liệu ban đầu: {X.shape}")
print("Các cột trong dữ liệu:")
print(X.columns.tolist())

# Tiền xử lý dữ liệu
print("\nĐang tiền xử lý dữ liệu...")

# Chuyển đổi InvoiceDate sang kiểu datetime
X['InvoiceDate'] = pd.to_datetime(X['InvoiceDate'], errors='coerce')

# Loại bỏ các hàng có InvoiceDate null
X = X.dropna(subset=['InvoiceDate'])

# Chuyển đổi các cột số và loại bỏ giá trị không hợp lệ
X['Quantity'] = pd.to_numeric(X['Quantity'], errors='coerce')
X['UnitPrice'] = pd.to_numeric(X['UnitPrice'], errors='coerce')
X = X.dropna(subset=['Quantity', 'UnitPrice'])

# THAY ĐỔI 1: Chỉ loại bỏ hàng có Quantity < 0 (giữ lại Quantity = 0)
X = X[X['Quantity'] >= 0]
X = X[X['UnitPrice'] >= 0]

# THAY ĐỔI MỚI: Xử lý StockCode - chuyển thành chữ in hoa và xóa khoảng trắng
print("\nĐang xử lý StockCode - chuyển thành chữ in hoa và xóa khoảng trắng...")
X['StockCode'] = X['StockCode'].astype(str).str.upper().str.replace(r'\s+', '', regex=True)

# THAY ĐỔI 2: Chuẩn hóa Description dựa trên StockCode
print("\nĐang chuẩn hóa Description dựa trên StockCode...")

# Tạo một bản sao của DataFrame ban đầu để giữ lại tất cả dữ liệu
X_original = X.copy()

# Lọc ra các bản ghi có Description không rỗng và có ý nghĩa để tạo bảng tra cứu
X_desc = X[~X['Description'].isna() & (X['Description'] != '')].copy()

print(f"Số lượng bản ghi có Description không rỗng: {X_desc.shape[0]}")

# Tạo DataFrame tần suất xuất hiện của mỗi cặp StockCode-Description
desc_counts = X_desc.groupby(['StockCode', 'Description']).size().reset_index(name='count')

# Sắp xếp theo StockCode và số lần xuất hiện (giảm dần)
desc_counts = desc_counts.sort_values(['StockCode', 'count'], ascending=[True, False])

# Lấy Description xuất hiện nhiều nhất cho mỗi StockCode
clean_descriptions = desc_counts.drop_duplicates(subset='StockCode', keep='first')[['StockCode', 'Description']]

print(f"Số lượng StockCode duy nhất có trong bảng tra cứu: {clean_descriptions.shape[0]}")

# Áp dụng Description chuẩn hóa cho toàn bộ DataFrame
# Loại bỏ cột Description hiện tại và join với cột Description đã chuẩn hóa
X_original = X_original.drop(columns=['Description'])
X_cleaned = pd.merge(X_original, clean_descriptions, on='StockCode', how='left')

# Kiểm tra xem có StockCode nào không có mô tả chuẩn hóa không
missing_desc = X_cleaned[X_cleaned['Description'].isna()]
print(f"Số lượng bản ghi không có mô tả sau khi join: {missing_desc.shape[0]}")

if missing_desc.shape[0] > 0:
    print("Một số StockCode không có Description chuẩn hóa. Đang gán giá trị mặc định...")
    # Điền giá trị mặc định cho các StockCode không có mô tả chuẩn hóa
    X_cleaned['Description'] = X_cleaned['Description'].fillna(f"Unknown Product")

X = X_cleaned

# Quan trọng: Chuyển đổi timestamp thành millisecond để tương thích với Spark
X['InvoiceDate'] = X['InvoiceDate'].dt.floor('ms')  # Làm tròn xuống millisecond

# Hiển thị dữ liệu mẫu sau xử lý
print("\nDữ liệu mẫu sau xử lý:")
print(X.head())

# Hiển thị thông tin về các StockCode có Description sau khi clean
print(f"\nSố lượng StockCode duy nhất trong dữ liệu sau khi xử lý: {X['StockCode'].nunique()}")
print(f"Số lượng bản ghi sau khi xử lý: {X.shape[0]}")

# Lưu thành file Parquet với cấu hình tương thích Spark
parquet_file = os.path.join(output_dir, 'associate_retail_spark.parquet')

print(f"\nĐang lưu file Parquet tại: {os.path.abspath(parquet_file)}")
X.to_parquet(
    parquet_file,
    engine='pyarrow',
    compression='snappy',
    index=False,
    coerce_timestamps='ms',         # Sử dụng millisecond cho timestamp
    allow_truncated_timestamps=True  # Cho phép làm tròn timestamp
)

# Hiển thị kích thước file
file_size_mb = os.path.getsize(parquet_file) / (1024 * 1024)
print(f"File Parquet đã được tạo thành công! Kích thước: {file_size_mb:.2f} MB")
print(f"Đường dẫn: {os.path.abspath(parquet_file)}")

# Đọc lại và kiểm tra file Parquet
print("\nĐọc lại file để kiểm tra:")
df_check = pd.read_parquet(parquet_file)
print(df_check.head())
print(f"Số lượng bản ghi: {len(df_check)}")
print(f"Kiểu dữ liệu InvoiceDate: {df_check['InvoiceDate'].dtype}")

# In thêm một số StockCode đã được xử lý để kiểm tra
print("\nMột số StockCode đã được xử lý (in hoa và xóa khoảng trắng):")
print(df_check['StockCode'].head(10).tolist())

Đang tải dữ liệu từ UCI ML Repository...
Kích thước dữ liệu ban đầu: (541909, 8)
Các cột trong dữ liệu:
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

Đang tiền xử lý dữ liệu...

Đang xử lý StockCode - chuyển thành chữ in hoa và xóa khoảng trắng...

Đang chuẩn hóa Description dựa trên StockCode...
Số lượng bản ghi có Description không rỗng: 530691
Số lượng StockCode duy nhất có trong bảng tra cứu: 3815
Số lượng bản ghi không có mô tả sau khi join: 14
Một số StockCode không có Description chuẩn hóa. Đang gán giá trị mặc định...

Dữ liệu mẫu sau xử lý:
  InvoiceNo StockCode  Quantity         InvoiceDate  UnitPrice  CustomerID  \
0    536365    85123A         6 2010-12-01 08:26:00       2.55     17850.0   
1    536365     71053         6 2010-12-01 08:26:00       3.39     17850.0   
2    536365    84406B         8 2010-12-01 08:26:00       2.75     17850.0   
3    536365    84029G         6 2010-12-01 08:26:00       3.39     178

### Reading parquet associate data

In [4]:
"""
Đọc file Parquet với PySpark và thực hiện phân tích dữ liệu
"""

import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, count, countDistinct, avg, desc, date_format
from pyspark.sql.functions import year, month, dayofmonth, hour, dayofweek

# Thiết lập JAVA_HOME (điều chỉnh đường dẫn cho phù hợp với máy của bạn)
os.environ['JAVA_HOME'] = r'C:\Program Files\Java\jdk-20'  # Thay đổi nếu cần

# Khởi tạo Spark Session
print("Khởi tạo Spark Session...")
spark = SparkSession.builder \
    .appName("OnlineRetailAnalysis") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

# Đường dẫn đến file Parquet
parquet_file = r"C:\Users\Admin\Desktop\big_data_project\associate_retail_data\associate_retail_spark.parquet"

# Kiểm tra file tồn tại
import os.path
if not os.path.exists(parquet_file):
    print(f"Lỗi: File không tồn tại tại {os.path.abspath(parquet_file)}")
    exit(1)
else:
    print(f"Đọc file Parquet từ: {os.path.abspath(parquet_file)}")

# Đọc file Parquet
print("\nĐọc file Parquet...")
try:
    df = spark.read.parquet(parquet_file)
    print("Đọc file thành công!")
except Exception as e:
    print(f"Lỗi khi đọc file: {str(e)}")
    spark.stop()
    exit(1)

# 1. Hiển thị schema và dữ liệu mẫu
print("\n=== SCHEMA DỮ LIỆU ===")
df.printSchema()

print("\n=== 5 DÒNG DỮ LIỆU MẪU ===")
df.show(5)

# 2. Thống kê cơ bản
print("\n=== THỐNG KÊ CƠ BẢN ===")
print(f"Số lượng bản ghi: {df.count()}")
print(f"Số lượng cột: {len(df.columns)}")
type(df)

Khởi tạo Spark Session...
Đọc file Parquet từ: C:\Users\Admin\Desktop\big_data_project\associate_retail_data\associate_retail_spark.parquet

Đọc file Parquet...
Đọc file thành công!

=== SCHEMA DỮ LIỆU ===
root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- InvoiceDate: timestamp_ntz (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: double (nullable = true)
 |-- Country: string (nullable = true)
 |-- Description: string (nullable = true)


=== 5 DÒNG DỮ LIỆU MẪU ===
+---------+---------+--------+-------------------+---------+----------+--------------+--------------------+
|InvoiceNo|StockCode|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|         Description|
+---------+---------+--------+-------------------+---------+----------+--------------+--------------------+
|   536365|   85123A|       6|2010-12-01 08:26:00|     2.55|   17850.0|United Kingdom|WHITE HANGING H

pyspark.sql.dataframe.DataFrame

### Processing data for associate analysis

In [12]:
from pyspark.sql import SparkSession
from pyspark.ml.fpm import FPGrowth
from pyspark.sql.functions import collect_set, col, size, count, struct, explode, array, lit, map_from_entries
import os
import pandas as pd

# Cài đặt HADOOP_HOME (quan trọng cho Windows)
os.environ['HADOOP_HOME'] = r'C:\hadoop'  # Thay đổi đường dẫn này tới thư mục cài đặt winutils.exe
os.environ['JAVA_HOME'] = r'C:\Program Files\Java\jdk-20'  # Thay đổi nếu cần

# Tạo thư mục đầu ra trước để tránh lỗi
import subprocess
import os
output_path = r"C:/Users/Admin/Desktop/big_data_project/associate_retail_data/fpgrowth_results_max30items"
os.makedirs(output_path, exist_ok=True)

# Khởi tạo SparkSession
spark = SparkSession.builder \
    .appName("Associate Analysis using FPGrowth") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

# Đọc file Parquet
df = spark.read.parquet("C:/Users/Admin/Desktop/big_data_project/associate_retail_data/associate_retail_spark.parquet")

# Hiển thị schema và một số dòng dữ liệu đầu tiên
print("Schema:")
df.printSchema()
print("\nSample Data:")
df.show(5, truncate=False)

# Tiền xử lý dữ liệu
# 1. Loại bỏ các dòng có số lượng âm (có thể là trả hàng)
df_filtered = df.filter(col("Quantity") > 0)

# 2. Tạo một mapping từ StockCode đến Description
# Lấy cặp StockCode-Description duy nhất
stockcode_desc_pairs = df_filtered.select("StockCode", "Description").distinct()
print("\nMẫu StockCode-Description:")
stockcode_desc_pairs.show(5, truncate=False)

# Tạo dictionary để mapping
stockcode_to_desc = {}
for row in stockcode_desc_pairs.collect():
    stockcode_to_desc[row["StockCode"]] = row["Description"]

# 3. Tập trung vào các cột InvoiceNo và StockCode
inv_items = df_filtered.select("InvoiceNo", "StockCode")

# 4. Nhóm theo hóa đơn để tạo giỏ hàng
# Quan trọng: Sử dụng collect_set thay vì collect_list để loại bỏ các mục trùng lặp
transactions = inv_items.groupBy("InvoiceNo").agg(
    collect_set("StockCode").alias("items")  # collect_set sẽ loại bỏ các phần tử trùng lặp
)

# Kiểm tra trùng lặp trong các items
print("\nKiểm tra mẫu sau khi loại bỏ trùng lặp:")
transactions.select("InvoiceNo", "items").show(5, truncate=False)

# Đếm số lượng giao dịch
total_txn_count = transactions.count()
print(f"\nTổng số giao dịch: {total_txn_count}")

# Tính kích thước giỏ hàng
transactions = transactions.withColumn("basket_size", size("items"))

# Hiển thị phân phối kích thước giỏ hàng
print("\nPhân phối kích thước giỏ hàng:")
basket_sizes = transactions.groupBy("basket_size").count().orderBy("basket_size")
basket_sizes.show(50)

# Lọc giỏ hàng có tối đa 30 món
filtered_transactions = transactions.filter(col("basket_size") <= 30)
filtered_txn_count = filtered_transactions.count()

print(f"\nSố giao dịch sau khi lọc (giỏ hàng <= 30 món): {filtered_txn_count}")
print(f"Đã loại bỏ {total_txn_count - filtered_txn_count} giao dịch có trên 30 món")

# Hiển thị một số mẫu giao dịch đã lọc
print("\nMẫu giao dịch sau khi lọc:")
filtered_transactions.select("InvoiceNo", "items", "basket_size").show(5, truncate=False)

# Đảm bảo không còn NULL trong dữ liệu
print("\nKiểm tra NULL trong items:")
null_count = filtered_transactions.filter(col("items").isNull()).count()
print(f"Số giao dịch có items là NULL: {null_count}")

if null_count > 0:
    print("Loại bỏ các giao dịch có items là NULL")
    filtered_transactions = filtered_transactions.filter(col("items").isNotNull())

# Chỉ lấy các cột cần thiết cho FP Growth
fp_input = filtered_transactions.select("items")

# Áp dụng thuật toán FP Growth
try:
    # Thử với minSupport và minConfidence ban đầu
    min_support = 0.007  # 0.7%
    min_confidence = 0.6  # 60%
    
    print(f"\nÁp dụng FP Growth với minSupport={min_support}, minConfidence={min_confidence}")
    fpGrowth = FPGrowth(itemsCol="items", 
                        minSupport=min_support,
                        minConfidence=min_confidence)
    
    # Huấn luyện mô hình
    model = fpGrowth.fit(fp_input)
    
    # Lấy các tập phổ biến
    frequent_itemsets = model.freqItemsets
    freq_count = frequent_itemsets.count()
    
    if freq_count == 0:
        # Nếu không có tập phổ biến, thử giảm minSupport
        print("Không tìm thấy tập phổ biến nào. Giảm minSupport xuống 0.003")
        min_support = 0.003  # 0.3%
        fpGrowth = FPGrowth(itemsCol="items", 
                            minSupport=min_support,
                            minConfidence=min_confidence)
        model = fpGrowth.fit(fp_input)
        frequent_itemsets = model.freqItemsets
        freq_count = frequent_itemsets.count()
    
    print(f"\nSố lượng tập phổ biến tìm thấy: {freq_count}")
    
    # Hiển thị các tập phổ biến đầu tiên
    print("\nCác tập phổ biến đầu tiên:")
    frequent_itemsets.show(20, truncate=False)
    
    # Lấy các luật kết hợp
    association_rules = model.associationRules
    rule_count = association_rules.count()
    
    print(f"\nSố lượng luật kết hợp tìm thấy: {rule_count}")
    
    # Hiển thị các luật kết hợp đầu tiên
    print("\nCác luật kết hợp đầu tiên:")
    association_rules.show(20, truncate=False)
    
    # Thêm mô tả vào các tập phổ biến
    # Đầu tiên, chuyển đổi frequent_itemsets thành pandas DataFrame
    freq_items_pd = frequent_itemsets.toPandas()
    
    # Thêm cột mô tả cho từng mã hàng
    def add_description(items_list):
        desc_list = []
        for item in items_list:
            desc = stockcode_to_desc.get(item, "Không có mô tả")
            desc_list.append(f"{item}: {desc}")
        return desc_list
    
    # Áp dụng hàm để thêm mô tả
    freq_items_pd['items_with_desc'] = freq_items_pd['items'].apply(add_description)
    
    # Lưu kết quả ra CSV với mô tả
    # Tạo thư mục đầu ra
    output_path = "C:/Users/Admin/Desktop/big_data_project/associate_retail_data/fpgrowth_results_max30items"
    os.makedirs(output_path, exist_ok=True)
    
    # Lưu frequent itemsets vào CSV với mô tả
    csv_freq_path = f"{output_path}/frequent_itemsets_with_desc.csv"
    print(f"\nLưu frequent itemsets vào {csv_freq_path}")
    freq_items_pd.to_csv(csv_freq_path, index=False)
    
    # Thêm mô tả vào các luật kết hợp
    rules_pd = association_rules.toPandas()
    
    # Thêm mô tả cho antecedent
    rules_pd['antecedent_with_desc'] = rules_pd['antecedent'].apply(add_description)
    
    # Thêm mô tả cho consequent
    rules_pd['consequent_with_desc'] = rules_pd['consequent'].apply(add_description)
    
    # Lưu association rules vào CSV với mô tả
    csv_rules_path = f"{output_path}/association_rules_with_desc.csv"
    print(f"Lưu association rules vào {csv_rules_path}")
    rules_pd.to_csv(csv_rules_path, index=False)
    
    # Lấy top 20 luật kết hợp theo độ tin cậy
    top_rules_pd = rules_pd.sort_values('confidence', ascending=False).head(20)
    
    # Lưu top rules vào CSV với mô tả
    csv_top_rules_path = f"{output_path}/top_association_rules_with_desc.csv"
    print(f"Lưu top rules vào {csv_top_rules_path}")
    top_rules_pd.to_csv(csv_top_rules_path, index=False)
    
    # Tạo một bản báo cáo thân thiện hơn (dễ đọc hơn) cho các luật kết hợp hàng đầu
    friendly_rules = []
    for idx, row in top_rules_pd.iterrows():
        antecedent_items = [f"{item}" for item in row['antecedent']]
        consequent_items = [f"{item}" for item in row['consequent']]
        
        antecedent_descs = [stockcode_to_desc.get(item, "Không có mô tả") for item in row['antecedent']]
        consequent_descs = [stockcode_to_desc.get(item, "Không có mô tả") for item in row['consequent']]
        
        friendly_rules.append({
            'rule_id': idx + 1,
            'antecedent_codes': ', '.join(antecedent_items),
            'antecedent_descriptions': '; '.join(antecedent_descs),
            'consequent_codes': ', '.join(consequent_items),
            'consequent_descriptions': '; '.join(consequent_descs),
            'confidence': row['confidence'],
            'lift': row['lift']
        })
    
    friendly_rules_df = pd.DataFrame(friendly_rules)
    
    # Lưu báo cáo thân thiện
    csv_friendly_path = f"{output_path}/friendly_top_rules.csv"
    print(f"Lưu báo cáo thân thiện vào {csv_friendly_path}")
    friendly_rules_df.to_csv(csv_friendly_path, index=False)
    
    # Hiển thị thông tin tóm tắt
    print(f"\nTổng kết:")
    print(f"- Tổng số giao dịch ban đầu: {total_txn_count}")
    print(f"- Số giao dịch được sử dụng (giỏ hàng <= 30 món): {filtered_txn_count}")
    print(f"- Số lượng tập phổ biến: {freq_count}")
    print(f"- Số lượng luật kết hợp: {rule_count}")
    print(f"- Với minSupport = {min_support}, minConfidence = {min_confidence}")
    
except Exception as e:
    print(f"\nGặp lỗi trong quá trình áp dụng FP Growth: {e}")
    print("\nThử với giá trị minSupport và minConfidence thấp hơn")
    
    try:
        # Thử lại với minSupport thấp hơn
        min_support = 0.001  # 0.1%
        min_confidence = 0.3  # 30%
        
        print(f"\nÁp dụng FP Growth với minSupport={min_support}, minConfidence={min_confidence}")
        fpGrowth = FPGrowth(itemsCol="items", 
                            minSupport=min_support,
                            minConfidence=min_confidence)
        
        # Huấn luyện mô hình
        model = fpGrowth.fit(fp_input)
        
        # Lấy các tập phổ biến
        frequent_itemsets = model.freqItemsets
        freq_count = frequent_itemsets.count()
        
        print(f"\nSố lượng tập phổ biến tìm thấy: {freq_count}")
        
        # Hiển thị các tập phổ biến đầu tiên
        print("\nCác tập phổ biến đầu tiên:")
        frequent_itemsets.show(20, truncate=False)
        
        # Lấy các luật kết hợp
        association_rules = model.associationRules
        rule_count = association_rules.count()
        
        print(f"\nSố lượng luật kết hợp tìm thấy: {rule_count}")
        
        # Hiển thị các luật kết hợp đầu tiên
        print("\nCác luật kết hợp đầu tiên:")
        association_rules.show(20, truncate=False)
        
        # Thêm mô tả vào các tập phổ biến
        # Đầu tiên, chuyển đổi frequent_itemsets thành pandas DataFrame
        freq_items_pd = frequent_itemsets.toPandas()
        
        # Thêm cột mô tả cho từng mã hàng
        freq_items_pd['items_with_desc'] = freq_items_pd['items'].apply(add_description)
        
        # Lưu kết quả ra CSV với mô tả
        output_path = "C:/Users/Admin/Desktop/big_data_project/associate_retail_data/fpgrowth_results_max30items_low_support"
        os.makedirs(output_path, exist_ok=True)
        
        # Lưu frequent itemsets vào CSV với mô tả
        csv_freq_path = f"{output_path}/frequent_itemsets_with_desc.csv"
        print(f"\nLưu frequent itemsets vào {csv_freq_path}")
        freq_items_pd.to_csv(csv_freq_path, index=False)
        
        # Thêm mô tả vào các luật kết hợp
        rules_pd = association_rules.toPandas()
        
        # Thêm mô tả cho antecedent
        rules_pd['antecedent_with_desc'] = rules_pd['antecedent'].apply(add_description)
        
        # Thêm mô tả cho consequent
        rules_pd['consequent_with_desc'] = rules_pd['consequent'].apply(add_description)
        
        # Lưu association rules vào CSV với mô tả
        csv_rules_path = f"{output_path}/association_rules_with_desc.csv"
        print(f"Lưu association rules vào {csv_rules_path}")
        rules_pd.to_csv(csv_rules_path, index=False)
        
        # Lấy top 20 luật kết hợp theo độ tin cậy
        top_rules_pd = rules_pd.sort_values('confidence', ascending=False).head(20)
        
        # Lưu top rules vào CSV với mô tả
        csv_top_rules_path = f"{output_path}/top_association_rules_with_desc.csv"
        print(f"Lưu top rules vào {csv_top_rules_path}")
        top_rules_pd.to_csv(csv_top_rules_path, index=False)
        
        # Tạo một bản báo cáo thân thiện hơn cho các luật kết hợp hàng đầu
        friendly_rules = []
        for idx, row in top_rules_pd.iterrows():
            antecedent_items = [f"{item}" for item in row['antecedent']]
            consequent_items = [f"{item}" for item in row['consequent']]
            
            antecedent_descs = [stockcode_to_desc.get(item, "Không có mô tả") for item in row['antecedent']]
            consequent_descs = [stockcode_to_desc.get(item, "Không có mô tả") for item in row['consequent']]
            
            friendly_rules.append({
                'rule_id': idx + 1,
                'antecedent_codes': ', '.join(antecedent_items),
                'antecedent_descriptions': '; '.join(antecedent_descs),
                'consequent_codes': ', '.join(consequent_items),
                'consequent_descriptions': '; '.join(consequent_descs),
                'confidence': row['confidence'],
                'lift': row['lift']
            })
        
        friendly_rules_df = pd.DataFrame(friendly_rules)
        
        # Lưu báo cáo thân thiện
        csv_friendly_path = f"{output_path}/friendly_top_rules.csv"
        print(f"Lưu báo cáo thân thiện vào {csv_friendly_path}")
        friendly_rules_df.to_csv(csv_friendly_path, index=False)
        
        # Hiển thị thông tin tóm tắt
        print(f"\nTổng kết:")
        print(f"- Tổng số giao dịch ban đầu: {total_txn_count}")
        print(f"- Số giao dịch được sử dụng (giỏ hàng <= 30 món): {filtered_txn_count}")
        print(f"- Số lượng tập phổ biến: {freq_count}")
        print(f"- Số lượng luật kết hợp: {rule_count}")
        print(f"- Với minSupport = {min_support}, minConfidence = {min_confidence}")
        
    except Exception as e:
        print(f"\nVẫn gặp lỗi khi thử lại: {e}")
        print("\nGợi ý:")
        print("1. Kiểm tra dữ liệu đầu vào")
        print("2. Kiểm tra phiên bản PySpark và các thư viện phụ thuộc")
        print("3. Thử giảm số lượng giỏ hàng bằng cách lọc thêm")

# Đóng SparkSession
spark.stop()

Schema:
root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- InvoiceDate: timestamp_ntz (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: double (nullable = true)
 |-- Country: string (nullable = true)
 |-- Description: string (nullable = true)


Sample Data:
+---------+---------+--------+-------------------+---------+----------+--------------+-----------------------------------+
|InvoiceNo|StockCode|Quantity|InvoiceDate        |UnitPrice|CustomerID|Country       |Description                        |
+---------+---------+--------+-------------------+---------+----------+--------------+-----------------------------------+
|536365   |85123A   |6       |2010-12-01 08:26:00|2.55     |17850.0   |United Kingdom|WHITE HANGING HEART T-LIGHT HOLDER |
|536365   |71053    |6       |2010-12-01 08:26:00|3.39     |17850.0   |United Kingdom|WHITE METAL LANTERN                |
|536365   |84406B   |8